# MD-GRTN v26 — PEMS08
**Target:** Beat MD-GRTN paper: MAE < 13.114 | RMSE < 22.623 | MAPE < 8.471%
**v25 problem:** MAE≈14.9 stuck — never beat baseline

## Root cause analysis & fixes

### FIX 1 (CRITICAL) — Dense adjacency
v25 Gaussian kernel on ALL 170×170 pairs → avg degree=168 → every node's neighbourhood identical → GCN/GAT outputted near-uniform features. **v26: only fill edges where actual road connections exist (avg degree=3.2).**

### FIX 2 (CRITICAL) — Wrong residual
v25 added `last_observed_flow` in normalised space as 12-step-ahead residual. Bad anchor. **v26: no residual — model predicts directly.**

### FIX 3 — TemporalTransformer encoding shape bug
v25 had repeated/incorrect `enc_t` computation. **v26: clean single computation: W(N,1)@E(1,S)→(N,S)→unsqueeze→tpe_proj→(N,S,d)→permute→(1,S,N,d).**

### Architecture changes
- d_model: 96→128 (better capacity)
- MDAF module added (lightweight multi-period denoising fusion)
- 3-matrix adjacency: A_bin + A_dist + A_dyna (learned)
- Temporal pooling decoder: Linear(S→P) + MLP
- CosineAnnealingWarmRestarts (no early plateau)
- Input noise augmentation during training


In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import pandas as pd

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f'Seed set: {seed} OK')

set_seed()
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
class Config:
    data_path = "/kaggle/input/pems08/PEMS08.npz"
    adj_path  = "/kaggle/input/pems08/PEMS08.csv"
    best_path = "mdgrtn_v26_best.pt"

    num_nodes   = 170
    in_features = 3
    seq_len     = 12
    pred_len    = 12
    feature_idx = 0
    train_ratio = 0.7
    val_ratio   = 0.1

    # Model — T4 safe
    d_model    = 128
    n_heads    = 4
    gcn_layers = 2
    st_layers  = 3
    dropout    = 0.15

    # Training
    batch_size   = 48
    lr           = 2e-3
    epochs       = 300
    patience     = 40
    weight_decay = 1e-3
    # Gaussian noise aug in normalised space
    # Paper adds std=10 to raw data; per-node std~146 → norm noise ~0.07
    noise_std    = 0.07

cfg    = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Config: d={cfg.d_model} | GCN={cfg.gcn_layers} | ST={cfg.st_layers} | '
      f'seq={cfg.seq_len} | batch={cfg.batch_size} | device={device}')

In [ ]:
def build_sparse_adj(csv_path, num_nodes):
    """
    Build sparse adjacency from actual road edges only.
    FIX vs v25: Gaussian kernel on ALL pairs gave avg degree=168
    (nearly fully connected, destroying spatial signal).
    Now: only fill weights where actual edges exist -> avg degree ~3.2.
    Returns: A_bin_norm, A_dist_norm  both shape (N,N) float32
    """
    try:
        df = pd.read_csv(csv_path, header=None, skiprows=1)
        df.columns = ['from', 'to', 'cost']
        df[['from','to']] = df[['from','to']].astype(int)
        df['cost']        = df['cost'].astype(float)
        sigma  = df['cost'].std()
        A_bin  = np.zeros((num_nodes, num_nodes), dtype=np.float32)
        A_dist = np.zeros((num_nodes, num_nodes), dtype=np.float32)
        for _, r in df.iterrows():
            i, j, c = int(r['from']), int(r['to']), float(r['cost'])
            w = float(np.exp(-(c**2) / (sigma**2 + 1e-8)))
            A_bin[i,j]  = A_bin[j,i]  = 1.0
            A_dist[i,j] = A_dist[j,i] = w
        def rn(A):
            s = A.sum(1, keepdims=True) + 1e-8
            return A / s
        nnz = int((A_bin > 0).sum())
        print(f'Sparse adj: edges={len(df)} nnz={nnz} avg_deg={nnz/num_nodes:.1f} sigma={sigma:.1f}')
        return rn(A_bin), rn(A_dist)
    except Exception as e:
        print(f'WARNING: adj failed ({e}), using identity')
        I = np.eye(num_nodes, dtype=np.float32)
        return I, I


def load_pems08(cfg):
    raw  = np.load(cfg.data_path)
    data = raw['data'].astype(np.float32)   # (T, N, 3)
    T, N, F = data.shape
    print(f'Data: {data.shape}')
    mean_np = data.mean(axis=0)
    std_np  = data.std(axis=0) + 1e-8
    data_norm = (data - mean_np) / std_np
    A_bin, A_dist = build_sparse_adj(cfg.adj_path, N)
    return data_norm, mean_np, std_np, A_bin, A_dist


class TrafficDataset(Dataset):
    def __init__(self, data_norm, seq_len, pred_len, feature_idx,
                 split_start=0, split_end=None, noise_std=0.0):
        self.data      = data_norm
        self.seq_len   = seq_len
        self.pred_len  = pred_len
        self.feat_idx  = feature_idx
        self.noise_std = noise_std
        T = len(data_norm)
        split_end = split_end or T
        self.indices = list(range(split_start, split_end - seq_len - pred_len + 1))

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        x = self.data[i : i+self.seq_len].copy()                          # (S,N,3)
        y = self.data[i+self.seq_len : i+self.seq_len+self.pred_len,
                      :, self.feat_idx].copy()                            # (P,N)
        if self.noise_std > 0:
            x += np.random.randn(*x.shape).astype(np.float32) * self.noise_std
        return torch.from_numpy(x), torch.from_numpy(y)


def build_dataloaders(cfg):
    set_seed()
    data_norm, mean_np, std_np, A_bin, A_dist = load_pems08(cfg)
    T  = len(data_norm)
    t1 = int(T * cfg.train_ratio)
    t2 = int(T * (cfg.train_ratio + cfg.val_ratio))
    kw    = dict(batch_size=cfg.batch_size, num_workers=2, pin_memory=True)
    ds_kw = dict(data_norm=data_norm, seq_len=cfg.seq_len,
                 pred_len=cfg.pred_len, feature_idx=cfg.feature_idx)
    dl_tr = DataLoader(TrafficDataset(**ds_kw, split_start=0,  split_end=t1,
                                      noise_std=cfg.noise_std), shuffle=True,  **kw)
    dl_va = DataLoader(TrafficDataset(**ds_kw, split_start=t1, split_end=t2), shuffle=False, **kw)
    dl_te = DataLoader(TrafficDataset(**ds_kw, split_start=t2, split_end=T),  shuffle=False, **kw)
    print(f'Train={len(dl_tr.dataset)} Val={len(dl_va.dataset)} Test={len(dl_te.dataset)}')
    return dl_tr, dl_va, dl_te, mean_np, std_np, A_bin, A_dist

print('Data utilities ready.')

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# INPUT PROJECTION  (B,S,N,F) -> (B,S,N,d)
# ──────────────────────────────────────────────────────────────────────
class InputProjection(nn.Module):
    def __init__(self, in_dim, d_model, dropout=0.1):
        super().__init__()
        self.proj  = nn.Linear(in_dim, d_model)
        self.norm  = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_model*2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model*2, d_model))
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):   # (B,S,N,F)
        h = self.drop(self.proj(x))
        h = self.norm(h)
        return self.norm2(h + self.ff(h))


# ──────────────────────────────────────────────────────────────────────
# MDAF MODULE — Multi-Period Diffusion Attention Fusion
# Approximates paper's MDAF: lightweight temporal Conv denoiser
# + multi-period cross-attention fusion.
# (B,S,N,d) -> (B,S,N,d)
# ──────────────────────────────────────────────────────────────────────
class ConvDenoiser(nn.Module):
    """Depthwise+pointwise Conv denoiser (approximates diffusion BackNet)."""
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.dw   = nn.Conv1d(d_model, d_model, kernel_size=3, padding=1, groups=d_model)
        self.pw   = nn.Conv1d(d_model, d_model, kernel_size=1)
        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):  # (BN, S', d)
        h = F.gelu(self.dw(x.transpose(1,2)))
        h = self.drop(self.pw(h)).transpose(1,2)
        return self.norm(x + h)


class MDAFModule(nn.Module):
    """Multi-Period Diffusion Attention Fusion. (B,S,N,d) -> (B,S,N,d)"""
    def __init__(self, d_model, n_heads, seq_len, dropout=0.1):
        super().__init__()
        self.s1 = max(1, seq_len // 3)
        self.s2 = max(1, 2*seq_len // 3)
        self.den1    = ConvDenoiser(d_model, dropout)
        self.den2    = ConvDenoiser(d_model, dropout)
        self.den3    = ConvDenoiser(d_model, dropout)
        self.attn    = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.combine = nn.Linear(d_model*3, d_model)
        self.norm    = nn.LayerNorm(d_model)
        self.drop    = nn.Dropout(dropout)

    def forward(self, x):  # (B,S,N,d)
        B, S, N, d = x.shape
        xf = x.permute(0,2,1,3).reshape(B*N, S, d)    # (BN,S,d)
        v1 = self.den1(xf[:, -self.s1:, :])
        v2 = self.den2(xf[:, -self.s2:, :])
        v3 = self.den3(xf)
        h1, _ = self.attn(xf, v1, v1)
        h2, _ = self.attn(xf, v2, v2)
        h3, _ = self.attn(xf, v3, v3)
        fused = self.drop(self.combine(torch.cat([h1, h2, h3], dim=-1)))
        out   = self.norm(xf + fused)
        return out.reshape(B, N, S, d).permute(0,2,1,3)  # (B,S,N,d)


# ──────────────────────────────────────────────────────────────────────
# MGRC MODULE — Multi-Graph Recurrent Convolution
# 3-matrix fused adjacency + stacked GCN+GRU blocks
# (B,S,N,d) -> (B,S,N,d)
# ──────────────────────────────────────────────────────────────────────
class MGRCBlock(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.gcn_w = nn.Linear(d_model, d_model, bias=False)
        self.gcn_b = nn.Parameter(torch.zeros(d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.gru   = nn.GRU(d_model, d_model, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_model*2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model*2, d_model))
        self.norm3 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, A):   # x:(B,S,N,d)  A:(N,N)
        B, S, N, d = x.shape
        xf    = x.reshape(B*S, N, d)
        neigh = torch.einsum('nm,bmd->bnd', A, xf)        # (BS,N,d)
        gcn   = F.relu(self.gcn_w(neigh) + self.gcn_b)
        x = self.norm1(x + self.drop(gcn.reshape(B,S,N,d)))
        xt        = x.permute(0,2,1,3).reshape(B*N, S, d)
        gru, _    = self.gru(xt)
        gru       = gru.reshape(B,N,S,d).permute(0,2,1,3)
        x = self.norm2(x + self.drop(gru))
        x = self.norm3(x + self.drop(self.ff(x)))
        return x


class MGRCModule(nn.Module):
    def __init__(self, d_model, num_nodes, num_layers=2, dropout=0.1):
        super().__init__()
        emb_dim       = 16
        self.E1       = nn.Parameter(torch.randn(num_nodes, emb_dim)*0.1)
        self.E2       = nn.Parameter(torch.randn(num_nodes, emb_dim)*0.1)
        self.adj_conv = nn.Conv2d(3, 1, kernel_size=1, bias=True)
        self.blocks   = nn.ModuleList([MGRCBlock(d_model, dropout) for _ in range(num_layers)])

    def fused_adj(self, A_bin, A_dist):
        A_dyna  = torch.softmax(F.relu(self.E1 @ self.E2.T), dim=-1)
        stack   = torch.stack([A_bin, A_dist, A_dyna], dim=0).unsqueeze(0)  # (1,3,N,N)
        A_F     = F.relu(self.adj_conv(stack).squeeze(0).squeeze(0))
        return A_F / (A_F.sum(-1, keepdim=True) + 1e-8)

    def forward(self, x, A_bin, A_dist):
        A = self.fused_adj(A_bin, A_dist)
        for blk in self.blocks:
            x = blk(x, A)
        return x


print('InputProjection + MDAFModule + MGRCModule defined.')

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# SPATIAL TRANSFORMER  — global cross-node attention at each timestep
# (B,S,N,d) -> (B,S,N,d)
# ──────────────────────────────────────────────────────────────────────
class SpatialTransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.W_spe = nn.Linear(d_model, d_model, bias=False)
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_model*4), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model*4, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, A):  # x:(B,S,N,d)  A:(N,N)
        B, S, N, d = x.shape
        xf  = x.reshape(B*S, N, d)
        spe = self.W_spe(torch.einsum('nm,bmd->bnd', A, xf))  # spatial PE
        xf  = xf + 0.1 * spe
        h, _ = self.attn(xf, xf, xf)
        xf   = self.norm1(xf + self.drop(h))
        xf   = self.norm2(xf + self.drop(self.ff(xf)))
        return xf.reshape(B, S, N, d)


# ──────────────────────────────────────────────────────────────────────
# TEMPORAL TRANSFORMER  — self-attention across S timesteps per node
# Sinusoidal PE + learnable hourly/daily/weekly encoding (paper Eq.21)
# (B,S,N,d) -> (B,S,N,d)
#
# Shape derivation (enc_t):
#   W_hour:(N,1)  E_hour:(1,S)  ->  W_hour@E_hour: (N,S)
#   sum of 3 -> enc:(N,S)
#   enc.unsqueeze(-1):(N,S,1)  -> tpe_proj:(N,S,d)
#   permute(1,0,2):(S,N,d) -> unsqueeze(0):(1,S,N,d)  [broadcasts over B]
# ──────────────────────────────────────────────────────────────────────
class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() *
                        (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):  # (BN,S,d)
        return x + self.pe[:, :x.size(1)]


class TemporalTransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout, seq_len, num_nodes):
        super().__init__()
        self.sin_pe   = SinusoidalPE(d_model)
        self.W_hour   = nn.Parameter(torch.randn(num_nodes, 1) * 0.02)
        self.W_day    = nn.Parameter(torch.randn(num_nodes, 1) * 0.02)
        self.W_week   = nn.Parameter(torch.randn(num_nodes, 1) * 0.02)
        t = torch.arange(seq_len).float()
        self.register_buffer('E_hour', (t % 12 + 1).unsqueeze(0))  # (1,S)
        self.register_buffer('E_day',  (t % 24 + 1).unsqueeze(0))
        self.register_buffer('E_week', (t % 7  + 1).unsqueeze(0))
        self.tpe_proj = nn.Linear(1, d_model)
        self.attn     = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ff       = nn.Sequential(
            nn.Linear(d_model, d_model*4), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model*4, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):  # (B,S,N,d)
        B, S, N, d = x.shape
        # Temporal position encoding -> (1,S,N,d)
        enc = (self.W_hour @ self.E_hour +    # (N,S)
               self.W_day  @ self.E_day  +
               self.W_week @ self.E_week)
        enc = self.tpe_proj(enc.unsqueeze(-1))      # (N,S,d)
        enc = enc.permute(1,0,2).unsqueeze(0)       # (1,S,N,d)
        x   = x + enc
        # Self-attn across S per node
        f    = x.permute(0,2,1,3).reshape(B*N, S, d)  # (BN,S,d)
        f    = self.sin_pe(f)
        h, _ = self.attn(f, f, f)
        h    = self.norm1(f + self.drop(h))
        h    = self.norm2(h + self.drop(self.ff(h)))
        return h.reshape(B,N,S,d).permute(0,2,1,3)    # (B,S,N,d)


# ──────────────────────────────────────────────────────────────────────
# STFORMER BLOCK  — Spatial -> add+norm -> Temporal  (paper Fig.5)
# ──────────────────────────────────────────────────────────────────────
class STFormerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout, seq_len, num_nodes):
        super().__init__()
        self.spatial  = SpatialTransformerLayer(d_model, n_heads, dropout)
        self.norm_mid = nn.LayerNorm(d_model)
        self.temporal = TemporalTransformerLayer(d_model, n_heads, dropout, seq_len, num_nodes)

    def forward(self, x, A):  # (B,S,N,d)
        ys = self.spatial(x, A)       # (B,S,N,d)
        x  = self.norm_mid(x + ys)    # paper Eq.20
        x  = self.temporal(x)         # (B,S,N,d)
        return x

print('SpatialTransformer + TemporalTransformer + STFormerBlock defined.')

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# FULL MODEL — MDGRTNv26
#
# Pipeline:
#   x(B,S,N,3)
#   -> InputProjection       (B,S,N,d)
#   -> MDAFModule            (B,S,N,d)   multi-period denoising+fusion
#   -> MGRCModule            (B,S,N,d)   GCN+GRU, 3-matrix fused adj
#   -> STFormerBlock x L     (B,S,N,d)   global spatial+temporal attn
#   -> Decoder               (B,P,N)
#
# Decoder:
#   (B,S,N,d) -> permute -> (B,N,d,S)
#   -> Linear(S,P)           (B,N,d,P)
#   -> permute               (B,P,N,d)
#   -> MLP(d->1)             (B,P,N)
# ──────────────────────────────────────────────────────────────────────
class MDGRTNv26(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        N, F, d, h, dr = cfg.num_nodes, cfg.in_features, cfg.d_model, cfg.n_heads, cfg.dropout
        S, P           = cfg.seq_len, cfg.pred_len

        self.input_proj = InputProjection(F, d, dr)
        self.mdaf       = MDAFModule(d, h, S, dr)
        self.mgrc       = MGRCModule(d, N, num_layers=cfg.gcn_layers, dropout=dr)
        self.stformer   = nn.ModuleList([
            STFormerBlock(d, h, dr, S, N) for _ in range(cfg.st_layers)])
        # Decoder
        self.temp_pool = nn.Linear(S, P)
        self.out_proj  = nn.Sequential(
            nn.LayerNorm(d),
            nn.Linear(d, d//2), nn.GELU(),
            nn.Dropout(dr),
            nn.Linear(d//2, 1))

    def forward(self, x_rec, A_bin, A_dist):  # x_rec:(B,S,N,3)
        x = self.input_proj(x_rec)         # (B,S,N,d)
        x = self.mdaf(x)                   # (B,S,N,d)
        x = self.mgrc(x, A_bin, A_dist)    # (B,S,N,d)
        for blk in self.stformer:
            x = blk(x, A_bin)              # (B,S,N,d)
        # Decoder
        B, S, N, d = x.shape
        x = x.permute(0,2,3,1)            # (B,N,d,S)
        x = self.temp_pool(x)             # (B,N,d,P)
        x = x.permute(0,3,1,2)            # (B,P,N,d)
        out = self.out_proj(x).squeeze(-1) # (B,P,N)
        return out


set_seed()
model = MDGRTNv26(cfg).to(device)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model ready | Parameters: {total:,}')
print(f'  d={cfg.d_model} | GCN={cfg.gcn_layers} | ST={cfg.st_layers}')

with torch.no_grad():
    dummy = torch.randn(2, cfg.seq_len, cfg.num_nodes, cfg.in_features).to(device)
    Ab    = torch.eye(cfg.num_nodes).to(device)
    out   = model(dummy, Ab, Ab)
print(f'Output: {out.shape}  expect [2, 12, 170]')
if torch.cuda.is_available():
    mem = torch.cuda.memory_allocated()/1e9
    print(f'GPU mem after forward: {mem:.3f} GB  (T4=15GB safe)')
    torch.cuda.empty_cache()

In [ ]:
def masked_mae(pred, true):
    mask = (true.abs() > 0.1).float()
    return (torch.abs(pred - true) * mask).sum() / (mask.sum() + 1e-8)


def huber_loss(pred, true, delta=0.5):
    err  = torch.abs(pred - true)
    loss = torch.where(err < delta, 0.5*err**2, delta*(err - 0.5*delta))
    return loss.mean()


def masked_rmse(pred, true):
    mask = (true.abs() > 0.1).float()
    mse  = ((pred-true)**2 * mask).sum() / (mask.sum() + 1e-8)
    return torch.sqrt(mse)


def masked_mape(pred, true, low_thresh=10.0):
    mask = (true.abs() > low_thresh).float()
    if mask.sum() < 1:
        return torch.tensor(0.0, device=pred.device)
    return (torch.abs((pred-true) / (true.abs() + 1e-8)) * mask).sum() / mask.sum() * 100

print('Metrics ready.')

In [ ]:
scaler = torch.amp.GradScaler('cuda')


def train_epoch(model, loader, optimizer, scheduler, A_bin, A_dist, device):
    model.train()
    total = 0.0
    for x_rec, y in loader:
        x_rec = x_rec.to(device, non_blocking=True)
        y     = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, A_bin, A_dist)
            loss = huber_loss(pred, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, A_bin, A_dist, device, mean_flow, std_flow):
    model.eval()
    maes, rmses, mapes = [], [], []
    for x_rec, y in loader:
        x_rec  = x_rec.to(device, non_blocking=True)
        y      = y.to(device, non_blocking=True)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, A_bin, A_dist)
        pred_d = pred.float() * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y.float()    * std_flow[None,None,:] + mean_flow[None,None,:]
        maes.append(masked_mae(pred_d, y_d).item())
        rmses.append(masked_rmse(pred_d, y_d).item())
        mapes.append(masked_mape(pred_d, y_d).item())
    return np.mean(maes), np.mean(rmses), np.mean(mapes)

print('Train/eval functions ready.')

In [ ]:
set_seed()
dl_train, dl_val, dl_test, mean_np, std_np, A_bin_np, A_dist_np = build_dataloaders(cfg)

mean_flow = torch.from_numpy(mean_np[:, cfg.feature_idx]).to(device)  # (N,)
std_flow  = torch.from_numpy(std_np [:, cfg.feature_idx]).to(device)  # (N,)
A_bin     = torch.from_numpy(A_bin_np).to(device)
A_dist    = torch.from_numpy(A_dist_np).to(device)

print(f'Train batches: {len(dl_train)} | Val: {len(dl_val)} | Test: {len(dl_test)}')
print(f'A_bin nnz={int((A_bin>0).sum())} A_dist nnz={int((A_dist>0).sum())}')

In [ ]:
set_seed()
model = MDGRTNv26(cfg).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

# CosineAnnealingWarmRestarts: avoids OneCycleLR early plateau
# T_0=50 -> first restart at ep50, then T_mult=2 doubles: 100, 200...
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=50, T_mult=2, eta_min=1e-5)

best_val_mae  = float('inf')
best_val_rmse = float('inf')
best_val_mape = float('inf')
patience_cnt  = 0
history       = {'train_loss': [], 'val_mae': [], 'val_rmse': [], 'val_mape': []}

print(f'MDGRTNv26 | d={cfg.d_model} | GCN={cfg.gcn_layers} | ST={cfg.st_layers}')
print(f'Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'lr={cfg.lr} | wd={cfg.weight_decay} | dropout={cfg.dropout} | batch={cfg.batch_size}')
print(f'Target -> MAE<13.114 | RMSE<22.623 | MAPE<8.471%')
print('='*65)

for epoch in range(1, cfg.epochs + 1):
    train_loss = train_epoch(
        model, dl_train, optimizer, scheduler, A_bin, A_dist, device)
    val_mae, val_rmse, val_mape = eval_epoch(
        model, dl_val, A_bin, A_dist, device, mean_flow, std_flow)

    history['train_loss'].append(train_loss)
    history['val_mae'].append(val_mae)
    history['val_rmse'].append(val_rmse)
    history['val_mape'].append(val_mape)

    tag = ''
    if val_mae < best_val_mae:
        best_val_mae  = val_mae
        best_val_rmse = val_rmse
        best_val_mape = val_mape
        patience_cnt  = 0
        torch.save(model.state_dict(), cfg.best_path)
        print(f'  Best saved -> {cfg.best_path}')
        tag = '  <- best'
    else:
        patience_cnt += 1
        if patience_cnt >= cfg.patience:
            print(f'Early stopping at epoch {epoch}.')
            break

    lr_now = optimizer.param_groups[0]['lr']
    if epoch % 5 == 0 or tag:
        print(f'Ep {epoch:03d} | Loss={train_loss:.4f} | '
              f'MAE={val_mae:.3f} RMSE={val_rmse:.3f} MAPE={val_mape:.2f}% '
              f'lr={lr_now:.1e}{tag}')

print(f'\nBest Val: MAE={best_val_mae:.3f} | RMSE={best_val_rmse:.3f} | MAPE={best_val_mape:.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], color='steelblue')
axes[0].set_title('Train Loss (Huber d=0.5)')
axes[0].set_xlabel('Epoch')

axes[1].plot(history['val_mae'], color='tab:orange', label='Val MAE')
axes[1].axhline(13.114, color='red', ls='--', label='Baseline 13.114')
axes[1].set_title('Validation MAE')
axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(history['val_rmse'], color='tab:green', label='Val RMSE')
axes[2].axhline(22.623, color='red', ls='--', label='Baseline 22.623')
axes[2].set_title('Validation RMSE')
axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
plt.savefig('training_curves_v26.png', dpi=150)
plt.show()

In [ ]:
model.load_state_dict(torch.load(cfg.best_path, map_location=device))

@torch.no_grad()
def paper_style_eval(model, loader, A_bin, A_dist, device, mean_flow, std_flow):
    model.eval()
    all_pred, all_true = [], []
    for x_rec, y in loader:
        x_rec  = x_rec.to(device)
        y      = y.to(device)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, A_bin, A_dist)
        pred_d = pred.float() * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y.float()    * std_flow[None,None,:] + mean_flow[None,None,:]
        all_pred.append(pred_d.cpu())
        all_true.append(y_d.cpu())

    P = torch.cat(all_pred, dim=0)
    Y = torch.cat(all_true, dim=0)
    mae  = torch.abs(P - Y).mean().item()
    rmse = ((P - Y)**2).mean().sqrt().item()
    mask = Y.abs() > 10.0
    mape = (torch.abs((P[mask]-Y[mask])/(Y[mask].abs()+1e-8))).mean().item()*100

    print('\n' + '='*60)
    print('  TEST RESULTS  -  MDGRTNv26  -  all 12 steps')
    print('='*60)
    print(f'  MAE  : {mae:.3f}   baseline: 13.114   d={mae-13.114:+.3f}')
    print(f'  RMSE : {rmse:.3f}   baseline: 22.623   d={rmse-22.623:+.3f}')
    print(f'  MAPE : {mape:.3f}%  baseline:  8.471%  d={mape-8.471:+.3f}%')
    beaten = (mae < 13.114) and (rmse < 22.623) and (mape < 8.471)
    print(f'\n  Baseline BEATEN: {"YES" if beaten else "Not yet"}')
    print('='*60)
    return mae, rmse, mape

mae, rmse, mape = paper_style_eval(
    model, dl_test, A_bin, A_dist, device, mean_flow, std_flow)

In [ ]:
@torch.no_grad()
def horizon_eval(model, loader, A_bin, A_dist, device, mean_flow, std_flow):
    model.eval()
    buf = {h: {'mae':[],'rmse':[],'mape':[]} for h in [2,5,11]}
    for x_rec, y in loader:
        x_rec  = x_rec.to(device)
        y      = y.to(device)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, A_bin, A_dist)
        pred_d = pred.float()*std_flow[None,None,:]+mean_flow[None,None,:]
        y_d    = y.float()   *std_flow[None,None,:]+mean_flow[None,None,:]
        for h in buf:
            buf[h]['mae'].append(masked_mae(pred_d[:,h,:], y_d[:,h,:]).item())
            buf[h]['rmse'].append(masked_rmse(pred_d[:,h,:], y_d[:,h,:]).item())
            buf[h]['mape'].append(masked_mape(pred_d[:,h,:], y_d[:,h,:]).item())

    print(f"\n{'Horizon':>14} | {'MAE':>8} | {'RMSE':>8} | {'MAPE':>9}")
    print('-'*52)
    for h, lbl in zip([2,5,11],['3-step(15m)','6-step(30m)','12-step(60m)']):
        m = {k: np.mean(v) for k,v in buf[h].items()}
        print(f"{lbl:>14} | {m['mae']:>8.3f} | {m['rmse']:>8.3f} | {m['mape']:>8.2f}%")

horizon_eval(model, dl_test, A_bin, A_dist, device, mean_flow, std_flow)